In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json

prompts = [
 "A person is locked out of their house with no key. What are some possible solutions?",
 "Should social media platforms be allowed to collect personal data from users? Explain your reasoning.",
 "What might happen if humans could suddenly read each other's thoughts?",
 "What does the phrase 'actions speak louder than words' mean to you?",
 "How might artificial intelligence change education in the next 20 years?",
 "Someone feels overwhelmed by their daily responsibilities. What strategies might help them manage their stress?",
 "Write a short story that begins with: 'The last person on Earth sat alone in a room. There was a knock on the door.'",
 "What are the potential benefits and drawbacks of remote work becoming the norm?",
 "What makes a life meaningful?",
 "How would you design a city that prioritizes both environmental sustainability and human happiness?"
]
model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

results = []
for prompt in prompts:
    # Calculate the prompt length
    prompt_tokens = tokenizer.encode(prompt)
    prompt_length = len(prompt_tokens)
    
    # Encode prompt and generate response
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=100)
    
    # Decode the full generated text
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract the response by removing the original prompt part if it was included
    if full_text.startswith(prompt):
        response = full_text[len(prompt):].strip()
    else:
        response = full_text.strip()
    
    # Calculate response token count
    response_length = len(tokenizer.encode(response))
    
    # Create a dictionary for this prompt-response pair
    results.append({
        "prompt": prompt,
        "response": response,
        "prompt_length": prompt_length,
        "response_length": response_length
    })

json_contents = {"entries": results}
# Save the results to a JSON file
with open("./kickstart.json", "w+") as f:
    json.dump(json_contents, f, indent=2)